# NB02 — Complexity Classifier Fine-tuning
**Model:** `microsoft/mdeberta-v3-base` → 3-class classifier (A / B / C)  
**Output:** Fine-tuned checkpoint pushed to HuggingFace Hub  
**Kaggle Dataset Input:** `crosslingual-rag-data`  
**Persistent Storage:** HF Hub (model) + `crosslingual-rag-results` (eval JSONs)

---
### Label Schema
- **A** — Simple factoid (answer ≤ 3 words, named entity / number)  
- **B** — Medium (answer > 3 words, single span)  
- **C** — Complex / multi-hop (all HotpotQA, hardcoded)

### Architecture Note
`microsoft/mdeberta-v3-base` has `hidden_size=768`, **not 384**.  
We use `AutoModelForSequenceClassification` — it builds the correct classification head automatically.

## 0. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers accelerate datasets scikit-learn huggingface_hub sentencepiece protobuf

## 1. Secrets & Config

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# ── EDIT THESE ──────────────────────────────────────────────────
HF_USERNAME    = "melisaolivia18"   # your HuggingFace username
HF_REPO_NAME   = "mdeberta-complexity-crosslingual"  # repo to push to
HF_REPO_ID     = f"{HF_USERNAME}/{HF_REPO_NAME}"
PRIVATE_REPO   = True
# ────────────────────────────────────────────────────────────────

DATA_DIR    = "/kaggle/input/datasets/melisaolivia/crosslingual-rag-data"
OUTPUT_DIR  = "/kaggle/working/complexity_classifier"
RESULTS_DIR = "/kaggle/working/results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Target HF repo : {HF_REPO_ID}")
print(f"Data dir       : {DATA_DIR}")
print(f"HF token loaded: {'yes' if HF_TOKEN else 'NO — check Kaggle Secrets'}")

## 2. Global Imports & Reproducibility

In [ ]:
import json, random, re, gc
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Load Raw Data

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

print("Loading datasets...")
hotpotqa_train = load_jsonl(f"{DATA_DIR}/hotpotqa_train.jsonl")
tydiqa_train   = load_jsonl(f"{DATA_DIR}/tydiqa_id_train.jsonl")
xquad_parallel = load_jsonl(f"{DATA_DIR}/xquad_id_parallel.jsonl")

print(f"HotpotQA train : {len(hotpotqa_train):,} instances")
print(f"TyDiQA-ID train: {len(tydiqa_train):,} instances")
print(f"XQuAD-ID eval  : {len(xquad_parallel):,} instances (eval only — NOT used for training)")

# Peek at field names
print("\nHotpotQA sample keys :", list(hotpotqa_train[0].keys()))
print("TyDiQA sample keys   :", list(tydiqa_train[0].keys()))

## 4. Labeling Pipeline

### 4.1 HotpotQA → All Class C (hardcoded)
No annotation needed — multi-hop by design.

### 4.2 TyDiQA-ID → Heuristic A / B labeling
```
Label A : answer word count ≤ 3  AND  answer looks like named entity or number
Label B : answer word count > 3  AND  answer is a phrase/clause (single span)
Discarded: anything that doesn't clearly fit A or B
```
TyDiQA-ID produces **no Class C** from this pipeline.

In [ ]:
import re
# ── Named entity / number heuristic (conservative) ────────────────
NUMBER_RE = re.compile(r"^[\d\.,\-\+\%\/\s]+$")
MEASUREMENT_RE = re.compile(r"^[\d\.,]+\s*[a-zA-Z²³%°km²]+\d*$", re.IGNORECASE)
BULAN_ID = {
    'januari','februari','maret','april','mei','juni',
    'juli','agustus','september','oktober','november','desember'
}


def looks_like_named_entity_or_number(text: str) -> bool:
    text = text.strip()
    if NUMBER_RE.match(text):
        return True
    words = text.split()
    if len(words) == 0:
        return False
    # Majority capitalised → named entity
    capitalised = sum(1 for w in words if w[0].isupper())
    return (capitalised / len(words)) >= TITLE_WORDS_THRESHOLD


def looks_like_named_entity_or_number(text: str) -> bool:
    text = text.strip()
    if not text:
        return False

    # 1. Pure number / decimal / fraction
    if NUMBER_RE.match(text):
        return True

    # 2. Measurement with unit: "20,04km2", "570 M", "1.004,132 km2"
    if MEASUREMENT_RE.match(text):
        return True
    tokens = text.split()
    if (len(tokens) == 2
            and re.fullmatch(r'[\d\.,]+', tokens[0])
            and re.fullmatch(r'[a-zA-Z²³%°]+\d*', tokens[1])):
        return True

    # 3. Date: contains a digit AND an Indonesian month name
    lower = text.lower()
    if any(m in lower for m in BULAN_ID) and re.search(r'\d', text):
        return True

    # 4. Standalone year or year range: "1998", "2001-2005"
    if re.fullmatch(r'\d{3,4}([\/\-]\d{2,4})?', text):
        return True

    # 5. Majority-capitalised tokens (named entity proxy) — unchanged from before
    words = text.split()
    capitalised = sum(1 for w in words if w and w[0].isupper())
    if len(words) > 0 and (capitalised / len(words)) >= 0.6:
        return True

    # 6. Single alphabetic token — short categorical answer ("tunggal", "jantan")
    if len(tokens) == 1 and tokens[0].isalpha():
        return True

    return False


# ── Apply labeling ─────────────────────────────────────────────────
def label_tydiqa_record(record):
    """
    Returns (label_str, record) or (None, record) if discarded.
    """
    answer_raw = record.get("answer", "")
    if isinstance(answer_raw, list):
        answer_raw = " ".join(answer_raw)
    answer = answer_raw.strip()

    if not answer:
        return None, record

    words = answer.split()
    word_count = len(words)

    if word_count <= 3 and looks_like_named_entity_or_number(answer):
        return "A", record
    elif word_count > 3:
        return "B", record
    else:
        # short but no NE signal → assign B, not discard
        return "B", record


labeled_tydiqa = []
discarded_tydiqa = 0

for rec in tydiqa_train:
    label, rec = label_tydiqa_record(rec)
    if label is None:
        discarded_tydiqa += 1
        continue

    q = rec.get("question", "")
    if not q:
        discarded_tydiqa += 1
        continue

    labeled_tydiqa.append({"text": q, "label": label, "lang": "id", "source": "tydiqa"})

print(f"TyDiQA-ID labeled : {len(labeled_tydiqa):,} instances")
print(f"TyDiQA-ID discarded: {discarded_tydiqa:,} instances")

In [ ]:
# ── HotpotQA → Class C ────────────────────────────────────────────
labeled_hotpotqa = []
for rec in hotpotqa_train:
    q = rec.get("question", "")
    if q:
        labeled_hotpotqa.append({"text": q, "label": "C", "lang": "en", "source": "hotpotqa"})

print(f"HotpotQA Class C   : {len(labeled_hotpotqa):,} instances")

# ── Merge ─────────────────────────────────────────────────────────
all_labeled = labeled_hotpotqa + labeled_tydiqa
random.shuffle(all_labeled)

label_counts = Counter(d["label"] for d in all_labeled)
print(f"\nMerged dataset total: {len(all_labeled):,}")
for lbl in ["A", "B", "C"]:
    print(f"  Class {lbl}: {label_counts.get(lbl, 0):,}")

## 5. Train / Validation Split
Stratified 80/20 split **by class AND language** — ensures Class A and B appear in val set.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify key: class + language
strat_keys = [f"{d['label']}_{d['lang']}" for d in all_labeled]

train_data, val_data = train_test_split(
    all_labeled,
    test_size=0.20,
    random_state=SEED,
    stratify=strat_keys
)

print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")

for split_name, split in [("Train", train_data), ("Val", val_data)]:
    c = Counter(d["label"] for d in split)
    print(f"  {split_name} → A:{c.get('A',0):,}  B:{c.get('B',0):,}  C:{c.get('C',0):,}")

## 6. Tokenisation

`max_length=128` — sufficient for short questions, keeps T4 VRAM comfortable.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

MODEL_NAME = "microsoft/mdeberta-v3-base"
MAX_LEN    = 128

# Label encoding
LABEL2ID = {"A": 0, "B": 1, "C": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
print("Tokenizer loaded.")


class ComplexityDataset(Dataset):
    def __init__(self, records, tokenizer, max_len):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(
            rec["text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "token_type_ids" : enc.get("token_type_ids", torch.zeros(self.max_len, dtype=torch.long)).squeeze(0),
            "labels"         : torch.tensor(LABEL2ID[rec["label"]], dtype=torch.long),
        }


BATCH_SIZE = 16

train_dataset = ComplexityDataset(train_data, tokenizer, MAX_LEN)
val_dataset   = ComplexityDataset(val_data,   tokenizer, MAX_LEN)

train_label_ids = [LABEL2ID[d["label"]] for d in train_data]
train_label_counts = Counter(train_label_ids)
sample_weights = [1.0 / train_label_counts[label_id] for label_id in train_label_ids]
train_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True
)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}")
print("Balanced sampler: enabled")

## 6.1 Training Diagnostics

## 7. Model Setup

**mDeBERTa-v3-base hidden size is 768** — `AutoModelForSequenceClassification` builds the head correctly.  
Weighted loss to compensate severe class imbalance (Class C >> A, B).

In [ ]:
def print_split_diagnostics(name, split):
    print(f"\n[{name}]")
    print("Class counts:", dict(Counter(d["label"] for d in split)))
    print("Source/lang by class:")
    for lbl in ["A", "B", "C"]:
        rows = [d for d in split if d["label"] == lbl]
        print(f"  {lbl}: source={dict(Counter(d['source'] for d in rows))} lang={dict(Counter(d['lang'] for d in rows))}")


print_split_diagnostics("Train", train_data)
print_split_diagnostics("Val", val_data)

val_counts = Counter(d["label"] for d in val_data)
majority_label, majority_n = val_counts.most_common(1)[0]
print(f"\nVal majority baseline: {majority_label} = {majority_n / len(val_data):.4f}")

print("\nSample texts per class:")
for lbl in ["A", "B", "C"]:
    print(f"\nClass {lbl}")
    for rec in [d for d in train_data if d["label"] == lbl][:5]:
        print(f"- [{rec['source']}/{rec['lang']}] {rec['text'][:180]}")

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    token=HF_TOKEN,
    torch_dtype=torch.float32,
    ignore_mismatched_sizes=True,
)
model = model.float().to(DEVICE)

# Verify actual hidden size (sanity check)
actual_hidden = model.config.hidden_size
print(f"Model loaded. Hidden size (confirmed): {actual_hidden}")
print(f"Classifier head: Linear({actual_hidden}, {NUM_LABELS})")
assert actual_hidden == 768, f"Unexpected hidden_size={actual_hidden} — check model config"

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable/1e6:.1f}M")

In [ ]:
# ── Weighted loss: w_i = total / (n_classes × n_i) ────────────────
train_labels = [LABEL2ID[d["label"]] for d in train_data]
label_counts_train = Counter(train_labels)
total_train = len(train_labels)

class_weights = []
for i in range(NUM_LABELS):
    n_i = label_counts_train.get(i, 1)  # avoid div/0
    w   = total_train / (NUM_LABELS * n_i)
    class_weights.append(w)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class weights (A, B, C):")
for lbl, w, n in zip(["A","B","C"], class_weights, [label_counts_train.get(i,0) for i in range(3)]):
    print(f"  Class {lbl}: weight={w:.4f}  (n={n:,})")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

def compute_loss(logits, labels):
    weight = class_weights_tensor.to(device=logits.device, dtype=logits.dtype)
    return torch.nn.functional.cross_entropy(logits, labels, weight=weight)

## 8. Training Loop

In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

NUM_EPOCHS    = 5
BACKBONE_LR   = 1e-5
CLASSIFIER_LR = 2e-5
WARMUP_RATIO  = 0.1

backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier.")]
classifier_params = [p for n, p in model.named_parameters() if n.startswith("classifier.")]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": classifier_params, "lr": CLASSIFIER_LR},
], weight_decay=0.01, eps=1e-6)

total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Training config:")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Backbone LR : {BACKBONE_LR}")
print(f"  Head LR     : {CLASSIFIER_LR}")
print(f"  Total steps : {total_steps:,}")
print(f"  Warmup steps: {warmup_steps:,}")
print(f"  Max seq len : {MAX_LEN}")

# Smoke test: first 20 batches must produce finite logits/loss before full training.
model.eval()
with torch.no_grad():
    for smoke_step, batch in enumerate(train_loader):
        if smoke_step >= 20:
            break
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE).view(-1).long()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.view(-1, NUM_LABELS)
        if not torch.isfinite(logits).all():
            raise FloatingPointError(f"Non-finite smoke-test logits at step={smoke_step}; labels={labels.detach().cpu().tolist()}")
        loss = compute_loss(logits, labels)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite smoke-test loss at step={smoke_step}; labels={labels.detach().cpu().tolist()}")
print("Smoke test passed: first 20 train batches have finite logits/loss")

In [ ]:
def assert_finite_tensor(name, tensor, labels=None, step=None, phase="train"):
    if torch.isfinite(tensor).all():
        return
    label_dump = labels.detach().cpu().tolist() if labels is not None else None
    raise FloatingPointError(f"Non-finite {name} during {phase} step={step}; labels={label_dump}")


def assert_finite_model(model, phase="train", step=None):
    for name, param in model.named_parameters():
        if not torch.isfinite(param).all():
            raise FloatingPointError(f"Non-finite parameter after {phase} step={step}: {name}")


def evaluate(model, loader, loss_fn, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for step, batch in enumerate(loader):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            # mDeBERTa-v3 does NOT use token_type_ids in its forward() signature
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits
            assert_finite_tensor("eval logits", logits, labels, step, phase="eval")
            if logits.size(-1) != NUM_LABELS:
                raise ValueError(f"Expected {NUM_LABELS} logits, got {logits.size(-1)}. Re-run the model setup cell.")
            labels = labels.view(-1).long()
            logits = logits.view(-1, NUM_LABELS)

            loss = compute_loss(logits, labels)
            assert_finite_tensor("eval loss", loss, labels, step, phase="eval")
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=[0, 1, 2], zero_division=0
    )
    avg_loss = total_loss / len(loader)

    return {
        "loss"     : avg_loss,
        "accuracy" : acc,
        "precision": prec.tolist(),
        "recall"   : rec.tolist(),
        "f1"       : f1.tolist(),
        "pred_dist": dict(Counter(ID2LABEL[p] for p in all_preds)),
        "preds"    : all_preds,
        "labels"   : all_labels,
    }


training_history = []
best_val_acc     = 0.0
best_epoch       = 0

print("Starting training...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", leave=True)
    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        # mDeBERTa-v3: do NOT pass token_type_ids
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        assert_finite_tensor("train logits", logits, labels, step, phase="train")
        if logits.size(-1) != NUM_LABELS:
            raise ValueError(f"Expected {NUM_LABELS} logits, got {logits.size(-1)}. Re-run the model setup cell.")
        labels = labels.view(-1).long()
        logits = logits.view(-1, NUM_LABELS)
        loss    = compute_loss(logits, labels)
        if not torch.isfinite(loss):
            print(f"Non-finite train loss at epoch={epoch}, step={step}; labels={labels.detach().cpu().tolist()}")
            optimizer.zero_grad(set_to_none=True)
            raise FloatingPointError("Stopping because train loss is non-finite")

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        if not torch.isfinite(grad_norm):
            optimizer.zero_grad(set_to_none=True)
            raise FloatingPointError(f"Non-finite grad norm at epoch={epoch}, step={step}")
        optimizer.step()
        assert_finite_model(model, phase="optimizer", step=step)
        scheduler.step()

        total_train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # ── Validation ──────────────────────────────────────────────
    val_results = evaluate(model, val_loader, loss_fn, DEVICE)

    print(f"\nEpoch {epoch} summary:")
    print(f"  Train loss : {avg_train_loss:.4f}")
    print(f"  Val loss   : {val_results['loss']:.4f}")
    print(f"  Val Acc    : {val_results['accuracy']:.4f}")
    for i, lbl in enumerate(["A","B","C"]):
        print(f"  Class {lbl}    P={val_results['precision'][i]:.3f}  R={val_results['recall'][i]:.3f}  F1={val_results['f1'][i]:.3f}")
    print(f"  Pred dist  : {val_results['pred_dist']}")
    print()

    epoch_log = {
        "epoch"          : epoch,
        "train_loss"     : avg_train_loss,
        "val_loss"       : val_results["loss"],
        "val_accuracy"   : val_results["accuracy"],
        "val_precision"  : val_results["precision"],
        "val_recall"     : val_results["recall"],
        "val_f1"         : val_results["f1"],
    }
    training_history.append(epoch_log)

    if val_results["accuracy"] > best_val_acc:
        best_val_acc = val_results["accuracy"]
        best_epoch   = epoch
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Best model saved (epoch {epoch}, val_acc={best_val_acc:.4f})")

print(f"\nTraining complete. Best epoch: {best_epoch}  Best val acc: {best_val_acc:.4f}")

# Check against IndoBERT baseline target
if best_val_acc >= 0.756:
    print(f"✓ Target met: {best_val_acc:.4f} ≥ 0.756 (IndoBERT baseline)")
else:
    print(f"⚠ Below target: {best_val_acc:.4f} < 0.756 — investigate class weights / data imbalance")

## 9. Save Classifier Eval Results

In [ ]:
from sklearn.metrics import confusion_matrix

# Reload best checkpoint before final eval
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR, torch_dtype=torch.float32).float().to(DEVICE)

final_val = evaluate(model, val_loader, loss_fn, DEVICE)
cm = confusion_matrix(final_val["labels"], final_val["preds"]).tolist()

classifier_eval = {
    "model"           : MODEL_NAME,
    "best_epoch"      : best_epoch,
    "val_accuracy"    : final_val["accuracy"],
    "val_precision"   : {"A": final_val["precision"][0], "B": final_val["precision"][1], "C": final_val["precision"][2]},
    "val_recall"      : {"A": final_val["recall"][0],    "B": final_val["recall"][1],    "C": final_val["recall"][2]},
    "val_f1"          : {"A": final_val["f1"][0],        "B": final_val["f1"][1],        "C": final_val["f1"][2]},
    "confusion_matrix": cm,
    "target_accuracy" : 0.756,
    "target_met"      : final_val["accuracy"] >= 0.756,
    "prediction_distribution": final_val["pred_dist"],
    "training_history": training_history,
    "hyperparameters" : {
        "num_epochs" : NUM_EPOCHS,
        "batch_size" : BATCH_SIZE,
        "max_length" : MAX_LEN,
        "backbone_lr": BACKBONE_LR,
        "classifier_lr": CLASSIFIER_LR,
        "balanced_sampler": True,
        "warmup_ratio"  : WARMUP_RATIO,
        "seed"       : SEED,
        "weighted_loss": True,
        "class_weights": {"A": class_weights[0], "B": class_weights[1], "C": class_weights[2]},
        "finite_guards": True,
    },
    "data_bias_note": "Class C comes from HotpotQA/en only; Class A/B come from TyDiQA/id only, so source/language are confounded with complexity labels.",
    "data_stats" : {
        "hotpotqa_train"      : len(labeled_hotpotqa),
        "tydiqa_labeled"      : len(labeled_tydiqa),
        "tydiqa_discarded"    : discarded_tydiqa,
        "total_train_split"   : len(train_data),
        "total_val_split"     : len(val_data),
    },
}

eval_path = f"{RESULTS_DIR}/classifier_eval.json"
with open(eval_path, "w") as f:
    json.dump(classifier_eval, f, indent=2)
print(f"Saved: {eval_path}")

print("\n=== Final Classifier Performance ===")
print(f"Accuracy : {final_val['accuracy']:.4f}")
for i, lbl in enumerate(["A","B","C"]):
    print(f"Class {lbl}  P={final_val['precision'][i]:.4f}  R={final_val['recall'][i]:.4f}  F1={final_val['f1'][i]:.4f}")
print(f"\nPrediction distribution: {final_val['pred_dist']}")
print("\nConfusion Matrix (rows=true, cols=pred):")
print("        A     B     C")
for i, row in zip(["A","B","C"], cm):
    print(f"  {i}   {row[0]:5d} {row[1]:5d} {row[2]:5d}")

## 10. Post-hoc Consistency Test on XQuAD-ID

Run the classifier on both EN and ID versions of the same questions.  
Compare per-pair predictions → consistency rate.  
Result reported in paper Table regardless of outcome.

In [ ]:
# Free VRAM before consistency test
torch.cuda.empty_cache()
gc.collect()

model.eval()

def predict_batch(texts, tokenizer, model, device, batch_size=32):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            outputs = model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
            )
        preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()
        all_preds.extend(preds)
    return all_preds


# Extract parallel EN/ID questions
# Expected keys from NB01: question_en, question_id_text (ID question), question_id (identifier)
# Fallback to common alternatives
def get_question(record, lang):
    if lang == "en":
        return record.get("question_en") or record.get("question", "")
    else:
        return record.get("question_id_text") or record.get("question_id") or record.get("question", "")

en_questions = [get_question(r, "en") for r in xquad_parallel]
id_questions = [get_question(r, "id") for r in xquad_parallel]

print(f"XQuAD-ID eval pairs: {len(xquad_parallel):,}")
print(f"Sample EN: {en_questions[0][:80]}")
print(f"Sample ID: {id_questions[0][:80]}")

print("\nRunning consistency test...")
en_preds = predict_batch(en_questions, tokenizer, model, DEVICE)
id_preds = predict_batch(id_questions, tokenizer, model, DEVICE)

# Per-pair consistency
consistent_pairs  = sum(1 for e, i in zip(en_preds, id_preds) if e == i)
consistency_rate  = consistent_pairs / len(xquad_parallel)

# Distribution breakdown
en_dist = Counter(ID2LABEL[p] for p in en_preds)
id_dist = Counter(ID2LABEL[p] for p in id_preds)

# Per-pair records for paper
pair_records = []
for idx, (rec, ep, ip) in enumerate(zip(xquad_parallel, en_preds, id_preds)):
    pair_records.append({
        "question_id"    : rec.get("question_id", idx),
        "pred_en"        : ID2LABEL[ep],
        "pred_id"        : ID2LABEL[ip],
        "consistent"     : ep == ip,
    })

consistency_results = {
    "n_pairs"            : len(xquad_parallel),
    "consistent_pairs"   : consistent_pairs,
    "consistency_rate"   : consistency_rate,
    "en_prediction_dist" : dict(en_dist),
    "id_prediction_dist" : dict(id_dist),
    "interpretation"     : (
        "High consistency → classifier generalizes cross-lingually. "
        "Low consistency → confirms linguistic bias; justified by translation-free architecture."
    ),
    "pairs"              : pair_records,
}

cons_path = f"{RESULTS_DIR}/classifier_consistency.json"
with open(cons_path, "w") as f:
    json.dump(consistency_results, f, indent=2)

print(f"\n=== Consistency Test Results ===")
print(f"Consistent pairs : {consistent_pairs:,} / {len(xquad_parallel):,}")
print(f"Consistency rate : {consistency_rate:.4f} ({consistency_rate*100:.1f}%)")
print(f"EN prediction dist: {dict(en_dist)}")
print(f"ID prediction dist: {dict(id_dist)}")
print(f"\nSaved: {cons_path}")

## 11. Push to HuggingFace Hub

Model checkpoint pushed to `{HF_REPO_ID}` (private repo).  
Loaded in NB03–NB08 via `AutoModelForSequenceClassification.from_pretrained(HF_REPO_ID, token=HF_TOKEN)`.

In [ ]:
from huggingface_hub import login, HfApi

login(token=HF_TOKEN)

# Create repo if it doesn't exist
api = HfApi()
try:
    api.create_repo(repo_id=HF_REPO_ID, private=PRIVATE_REPO, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo creation note: {e}")

# Push model + tokenizer
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"\n✓ Model pushed to: https://huggingface.co/{HF_REPO_ID}")
print("Usage in downstream notebooks:")
print(f"  from transformers import AutoModelForSequenceClassification, AutoTokenizer")
print(f"  model = AutoModelForSequenceClassification.from_pretrained('{HF_REPO_ID}', token=HF_TOKEN)")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{HF_REPO_ID}', token=HF_TOKEN)")

## 12. Upload Eval Results to Kaggle Dataset

Upload `classifier_eval.json` and `classifier_consistency.json` to  
`crosslingual-rag-results` Kaggle Dataset via the API.

> **If you prefer manual upload:** skip this cell and upload the files from  
> `/kaggle/working/results/` via the Kaggle UI "Save & Run All" → Datasets.

In [ ]:
import shutil

# Copy results to /kaggle/working/outputs for easy access
outputs_dir = "/kaggle/working/outputs"
os.makedirs(outputs_dir, exist_ok=True)

for fname in ["classifier_eval.json", "classifier_consistency.json", "nb02_labeling_log.json"]:
    src = f"{RESULTS_DIR}/{fname}"
    dst = f"{outputs_dir}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied: {dst}")

print("\nFiles ready in /kaggle/working/outputs/")
print("→ Attach this as output to Kaggle Dataset 'crosslingual-rag-results'")

## 13. Notebook Summary

Run this cell to print a final audit of all outputs.

In [ ]:
print("="*60)
print("NB02 — COMPLETION SUMMARY")
print("="*60)

print(f"\n[DATA]")
print(f"  HotpotQA instances (class C) : {len(labeled_hotpotqa):,}")
print(f"  TyDiQA-ID labeled (A/B)      : {len(labeled_tydiqa):,}")
print(f"  TyDiQA-ID discarded          : {discarded_tydiqa:,}")
print(f"  Internal train / val split   : {len(train_data):,} / {len(val_data):,}")

print(f"\n[MODEL]")
print(f"  Base model   : {MODEL_NAME}")
print(f"  Hidden size  : {model.config.hidden_size} (confirmed 768)")
print(f"  Num labels   : {NUM_LABELS}")
print(f"  HF repo      : {HF_REPO_ID}")

print(f"\n[RESULTS]")
print(f"  Best epoch   : {best_epoch}")
print(f"  Val accuracy : {best_val_acc:.4f}")
print(f"  Target (0.756): {'✓ MET' if best_val_acc >= 0.756 else '✗ NOT MET'}")
print(f"  Consistency  : {consistency_rate:.4f} ({consistency_rate*100:.1f}%)")

print(f"\n[OUTPUT FILES]")
for f in ["classifier_eval.json", "classifier_consistency.json", "nb02_labeling_log.json"]:
    path = f"{outputs_dir}/{f}"
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {'✓' if exists else '✗'} {f} ({size/1024:.1f} KB)")

print(f"\n[NEXT STEPS]")
print(f"  1. Add outputs to 'crosslingual-rag-results' Kaggle Dataset")
print(f"  2. Proceed to NB03 (retrieval indexing)")
print(f"  3. In NB05+: load classifier from HF Hub: '{HF_REPO_ID}'")
print("="*60)